# Семинар 5. Потоки (threads)

Лекция ввела `threading`: потоки, замки, условные переменные, очереди, GIL.
Здесь — то, что реально ломается: почему `wait` только в цикле `while`,
какая очередь есть «стандартная» и почему она MPMC, напоминание про GIL
(без погружения) и живые примеры дедлоков и лайвлоков.

| # | Раздел |
|---|---|
| 1 | Потоки за две минуты (recap) |
| 2 | `Condition`: инвариант, `while` против `if`, ложные пробуждения |
| 3 | Очереди: классификация SPSC…MPMC и зоопарк `queue` |
| 4 | GIL: напоминание — I/O против счёта |
| 5 | Дедлоки: устроить, найти, предотвратить |
| 6 | Лайвлоки и старвация |

Все демки детерминированы или отработают за считанные секунды —
зависших потоков в ноутбуке не останется.

In [ ]:
import faulthandler
import threading
import time
from queue import Queue

print("окружение готово")

## 1. Потоки за две минуты

Всё из лекции в одной ячейке: `Thread(target=..., args=...)` создаёт,
`start()` запускает, `join()` ждёт завершения. Потоки делят память —
поэтому всё удобное (`shared.append(x)`) одновременно и опасное.
Семинар про то, как жить с этим.

In [ ]:
def greet(name: str) -> None:
    print(f"поток {name}: стартовал")
    time.sleep(0.05)
    print(f"поток {name}: закончил")


threads = [threading.Thread(target=greet, args=(f"t{i}",)) for i in range(3)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print("main: все потоки завершены")

## 2. Условные переменные: инвариант и цикл `while`

`threading.Condition` — это мьютекс + очередь ожидания. `wait()` атомарно
отпускает мьютекс и паркует поток; `notify()` / `notify_all()` будят.

Ключевое слово — **инвариант**: предикат над разделяемым состоянием,
который должен выполняться, когда поток идёт дальше после `wait()`
(«буфер пуст» перед записью, «в буфере есть элемент» перед чтением).

Правило, которое нельзя нарушать никогда:

```python
while not предикат:
    cond.wait()
```

Почему не `if` — разбираем на двух примерах ниже.

In [ ]:
class OneSlot:
    """Буфер на один элемент, скоординированный через Condition."""

    def __init__(self) -> None:
        self._cond = threading.Condition()
        self._item: object | None = None

    def put(self, item: object) -> None:
        with self._cond:
            while self._item is not None:  # инвариант: буфер пуст
                self._cond.wait()
            self._item = item
            self._cond.notify_all()

    def get(self) -> object:
        with self._cond:
            while self._item is None:  # инвариант: в буфере есть элемент
                self._cond.wait()
            item = self._item
            self._item = None
            self._cond.notify_all()
            return item


# Прогон: производитель пишет 0..99, потребитель читает.
box = OneSlot()
got: list[object] = []


def producer() -> None:
    for i in range(100):
        box.put(i)


def consumer() -> None:
    for _ in range(100):
        got.append(box.get())


tp = threading.Thread(target=producer)
tc = threading.Thread(target=consumer)
tp.start()
tc.start()
tp.join()
tc.join()

assert got == list(range(100)), f"порядок нарушен: {got[:5]}..."
print(f"ping-pong из {len(got)} элементов прошёл чисто")

### Почему `if` недопустим: `notify_all` и ложные пробуждения

Два законных источника «проснулся — а инвариант не выполнен»:

1. **`notify_all()`**. Он будит *всех* ожидающих, а единица ресурса — одна.
   Второй проснувшийся обязан уйти обратно в `wait()`. С `if` он вместо
   этого поплывёт дальше с нарушенным инвариантом.
2. **Спящие (spurious) пробуждения**. Стандарт POSIX прямо разрешает `wait`
   вернуться без `notify` — CPython-потоки почти не страдают, но
   `threading.Condition` — обёртка над примитивами ОС, и полагаться на
   «будит только кто-то конкретный» нельзя. `queue.Queue` внутри CPython
   тоже пишет `while not ...: wait()`.

Демка ниже детерминированно инсценирует первый случай: два потребителя
ждут, производитель кладёт один элемент и делает `notify_all`. Счётчик
`_waiting` наращивается под тем же мьютексом, что и `wait`, поэтому
по `_waiting == 2` гарантированно: оба потребителя уже стоят в очереди
ожидания и `notify_all` их не потеряет.

In [ ]:
class BuggyOneSlot:
    """Тот же буфер, но проверка инварианта через `if` — баг."""

    def __init__(self) -> None:
        self._cond = threading.Condition()
        self._item: object | None = None
        self._waiting = 0

    def put(self, item: object) -> None:
        with self._cond:
            while self._item is not None:
                self._cond.wait()
            self._item = item
            self._cond.notify_all()

    def get(self) -> object:
        with self._cond:
            self._waiting += 1  # под мьютексом cond: ровно перед wait
            if self._item is None:  # <-- БАГ: надо while
                self._cond.wait()
            self._waiting -= 1
            item = self._item
            self._item = None
            self._cond.notify_all()
            return item

    def waiters(self) -> int:
        with self._cond:
            return self._waiting


buggy = BuggyOneSlot()
results: dict[int, object] = {}


def greedy_consumer(i: int) -> None:
    results[i] = buggy.get()


consumers = [
    threading.Thread(target=greedy_consumer, args=(i,), daemon=True) for i in range(2)
]
for t in consumers:
    t.start()

while buggy.waiters() < 2:  # оба точно внутри wait()
    time.sleep(0.001)

buggy.put("единственный элемент")
for t in consumers:
    t.join()

print("результаты:", results)
stolen_none = [i for i, v in results.items() if v is None]
print(f"потребитель(и), проснувшиеся вхолостую: {stolen_none}")
assert stolen_none, "повезло со планировкой — перезапусти ячейку"
print("один элемент -> два проснувшихся -> второй унёс None при `if`")

С `while` второй потребитель после пробуждения перепроверил бы
инвариант, увидел пустой буфер и вернулся бы в `wait()` — потерянного
элемента не случилось бы. Это и есть вся разница между `if` и `while`
на одну строчку кода.

Вывод, который надо унести с собой: **`wait()` — только внутри `while`
по предикату-инварианту**. Точка. Не «почти всегда», не «вдруг
повезёт» — всегда. `queue.Queue` из следующего раздела делает это
за вас.

## 3. Очереди: классификация

Очередь описывается тем, *кто* в неё пишет и *кто* из неё читает.
Обычных комбинаций четыре:

| | один потребитель | много потребителей |
|---|---|---|
| **один производитель** | SPSC | SPMC |
| **много производителей** | MPSC | MPMC |

SPSC — самая простая (достаточно одного события/семафора). MPMC —
самая общая: любое число тех и других, и синхронизировать её заметно
сложнее (нужны и «не пусто», и «не переполнено», и честное пробуждение
ровно одного читателя).

**Стандартная очередь Питона `queue.Queue` — именно MPMC**:
много производителей, много потребителей, потокобезопасно из коробки.
Внутри — мьютекс и две условные переменные (`not_empty` / `not_full`),
то есть всё то, что мы руками писали в `OneSlot`, только проверенное.

### Зоопарк очередей в Питоне

| класс | модель | кто может использовать | примечание |
|---|---|---|---|
| `queue.Queue` | FIFO, опционально ограниченная | **MPMC, потоки** | `task_done()` / `join()` для подсчёта обработки |
| `queue.LifoQueue` | LIFO (стек) | MPMC, потоки | тот же каркас |
| `queue.PriorityQueue` | min-куча | MPMC, потоки | элементы `(priority, item)` |
| `queue.SimpleQueue` | FIFO, неограниченная | потоки | упрощённая, без `task_done`/`join` |
| `asyncio.Queue` | FIFO | один event loop | **не thread-safe**: внутри один loop, нет блокировок |
| `multiprocessing.Queue` | FIFO | процессы | сериализация через pickle, другая цена операций |

Правило выбора: между потоками — `queue.Queue` (MPMC по умолчанию),
внутри asyncio-программы — `asyncio.Queue`, между процессами —
`multiprocessing.Queue`. Руками на `Condition` пишем очередь
только ради учёбы — как в разделе 2.

In [ ]:
# MPMC в деле: 3 производителя x 4 потребителя на одной queue.Queue.
work: Queue[int | None] = Queue(maxsize=8)  # ограничение = классика bounded buffer
collected: list[int] = []
collected_lock = threading.Lock()

N_PRODUCERS, N_CONSUMERS, PER_PRODUCER = 3, 4, 10


def make_items(pid: int) -> None:
    for i in range(PER_PRODUCER):
        work.put(pid * 100 + i)  # блокируется, если буфер полон


def eat_items() -> None:
    while True:
        item = work.get()
        if item is None:  # стоп-сигнал
            work.task_done()
            return
        with collected_lock:
            collected.append(item)
        work.task_done()


producers = [threading.Thread(target=make_items, args=(p,)) for p in range(N_PRODUCERS)]
consumers = [threading.Thread(target=eat_items, daemon=True) for _ in range(N_CONSUMERS)]
for t in producers + consumers:
    t.start()
for t in producers:
    t.join()
work.join()  # всё произведённое доехало и обработано
for _ in consumers:
    work.put(None)  # стоп-сигналы
for t in consumers:
    t.join()

expected = {p * 100 + i for p in range(N_PRODUCERS) for i in range(PER_PRODUCER)}
assert len(collected) == len(expected) and set(collected) == expected
print(f"MPMC: {N_PRODUCERS} производителей x {N_CONSUMERS} потребителей, "
      f"потерь нет, элементов: {len(collected)}")

## 4. GIL — напоминание (без погружения)

GIL (Global Interpreter Lock) — один мьютекс на интерпретатор: в каждый
момент байткод исполняет **один** поток. Но во время блокирующих операций
I/O (`sleep`, сеть, диск, многие C-расширения) интерпретатор **отпускает**
GIL — другой поток в это время работает.

Практический вывод из двух частей:

* **I/O-bound задачи** (запросы, файлы, БД) — потоки реально ускоряют.
* **CPU-bound счёт в чистом Питоне** — потоки не ускоряют (вредят);
  берём `multiprocessing` или C-расширения.

С 3.13 существуют free-threaded сборки без GIL, но семантика потоков
от этого не меняется: гонки, дедлоки и правило «инвариант в `while`»
остаются. Глубже — в лекции, здесь только замер.

In [ ]:
def spin(n: int) -> int:
    """Чистый счёт на Питоне — держит GIL все n итераций."""
    total = 0
    for i in range(n):
        total += i * i
    return total


N = 2_000_000

# CPU-bound: два потока против двух прогонов подряд
start = time.perf_counter()
spin(N)
spin(N)
serial_cpu = time.perf_counter() - start

start = time.perf_counter()
ts = [threading.Thread(target=spin, args=(N,)) for _ in range(2)]
for t in ts:
    t.start()
for t in ts:
    t.join()
threads_cpu = time.perf_counter() - start

# I/O-bound: sleep отпускает GIL
start = time.perf_counter()
time.sleep(0.2)
time.sleep(0.2)
serial_io = time.perf_counter() - start

start = time.perf_counter()
ts = [threading.Thread(target=time.sleep, args=(0.2,)) for _ in range(2)]
for t in ts:
    t.start()
for t in ts:
    t.join()
threads_io = time.perf_counter() - start

print(f"CPU: последовательно {serial_cpu:.2f} c, в 2 потоках {threads_cpu:.2f} c "
      f"(коэффициент {serial_cpu / threads_cpu:.2f} ≈ 1 — GIL счёт не ускоряет)")
print(f"I/O: последовательно {serial_io:.2f} c, в 2 потоках {threads_io:.2f} c "
      f"(коэффициент {serial_io / threads_io:.2f} ≈ 2 — GIL отпущен во время sleep)")

## 5. Дедлоки

Дедлок — цикл ожиданий: каждый участник держит ресурс и ждёт ресурс,
который держит другой участник цикла. Классика жанра — два замка,
захватываются в разном порядке двумя потоками.

Ниже — честный дедлок. Потоки daemon: мы их не ждём вечным `join`,
а через секунду смотрим `faulthandler`-дамп стеков: оба потока видно
внутри `acquire`. Так дедлоки и диагностируют в реальных программах —
`faulthandler.dump_traceback_later` ставят в проде, чтобы поймать,
кто на чём повис.

In [ ]:
lock_a = threading.Lock()
lock_b = threading.Lock()


def alice() -> None:
    with lock_a:
        time.sleep(0.05)  # гарантируем перекрёстный захват
        with lock_b:  # сюда не попадём никогда
            print("alice: оба замка")


def bob() -> None:
    with lock_b:
        time.sleep(0.05)
        with lock_a:  # и сюда тоже
            print("bob: оба замка")


t_alice = threading.Thread(target=alice, daemon=True)
t_bob = threading.Thread(target=bob, daemon=True)
t_alice.start()
t_bob.start()

# Через секунду печатаем стеки всех потоков — дедлок увиден.
faulthandler.dump_traceback_later(1.0)
t_alice.join(timeout=1.5)
t_bob.join(timeout=1.5)
faulthandler.cancel_dump_traceback_later()

print("alice жива:", t_alice.is_alive(), "| bob жив:", t_bob.is_alive())
print("-- в дампе выше оба потока должны стоять внутри acquire* --")

### Лечение дедлоков

1. **Фиксированный порядок захвата** (иерархия замков): все потоки берут
   замки в одном глобальном порядке — цикл ожиданий не образуется.
   Порядок «по `id(lock)`» — типовой приём, когда замки динамические.
2. **`acquire(timeout=...)`**: захват с таймаутом + откат уже взятых
   замков + ретрай. (Помнит про лайвлок из раздела 6!)
3. **Один замок вместо двух** — грубо, зато без циклов.
4. **Посредник-очередь**: потоки не делят замки напрямую, а общаются
   через `queue.Queue` — классика «actor-style»; встретится
   в следующем семестре подробно.

## 6. Лайвлоки и старвация

Лайвлок — «дедлок без сна»: потоки не заблокированы, они активно что-то
делают, но каждый их шаг отменяется другим. Классика — «вежливые
философы»: каждый берёт левую вилку, видит занятую правую, кладёт левую
обратно и пробует снова. Если ретраи синхронны, философы вечно
расходятся и встречаются в одном такте — есть никто не ест, все заняты.
В демке синхронность дают барьеры: коллизия гарантирована в каждом
раунде, результат не зависит от удачи планировщика.

Старвация — третий родственник: поток готов к работе, но планировщик
(или чужие захваты) постоянно обходят его. Лечится честностью
(очереди ожидания FIFO) и таймаутами.

| | состояние потоков | признак |
|---|---|---|
| дедлок | спят, ждут друг друга | CPU 0%, стеки в `acquire` |
| лайвлок | активны, шаги отменяются | CPU горячий, прогресса нет |
| старвация | готовы, но не получают ресурс | один голодает при работающих остальных |

In [ ]:
# Два "вежливых философа". Три барьера на раунд: старт, обе левые
# вилки взяты, оба отступили. Третий важен: без него тот, чей таймаут
# истёк первым, отпустил бы вилку, а чужой acquire успел бы её захватить.
forks = [threading.Lock(), threading.Lock()]
round_gate = threading.Barrier(2)  # старт раунда
held_gate = threading.Barrier(2)  # оба держат левые вилки
given_up_gate = threading.Barrier(2)  # оба отступили, левые ещё у обоих
stop = threading.Event()
attempts = [0, 0]
meals = 0
meals_lock = threading.Lock()


def polite(i: int) -> None:
    global meals
    left, right = forks[i], forks[1 - i]
    while not stop.is_set():
        round_gate.wait()  # оба стартуют раунд одновременно
        attempts[i] += 1
        with left:
            held_gate.wait()  # теперь сосед точно держит свою левую
            got_right = right.acquire(timeout=0.05)  # правая занята
            given_up_gate.wait()  # никто не отпустил левую раньше времени
            if got_right:
                try:
                    with meals_lock:
                        meals += 1
                finally:
                    right.release()
            # else: кладём левую и пробуем снова — вежливость


threads = [threading.Thread(target=polite, args=(i,), daemon=True) for i in range(2)]
for t in threads:
    t.start()
time.sleep(0.7)
stop.set()
for t in threads:
    t.join(timeout=1.0)

assert all(not t.is_alive() for t in threads)
print(f"лайвлок: попыток {sum(attempts)}, съедено {meals}")
assert meals == 0, "лайвлок не воспроизвёлся"
print("потоки крутились, но прогресс ровно 0 — это и есть лайвлок")

# Лечение: общий порядок захвата (иерархия замков) вместо вежливости.
stop2 = threading.Event()
meals2 = 0


def ordered(i: int) -> None:
    global meals2
    first, second = forks[0], forks[1]  # порядок один для всех
    while not stop2.is_set():
        with first:
            with second:
                meals2 += 1


threads = [threading.Thread(target=ordered, args=(i,), daemon=True) for i in range(2)]
for t in threads:
    t.start()
time.sleep(0.5)
stop2.set()
for t in threads:
    t.join(timeout=1.0)

assert all(not t.is_alive() for t in threads)
print(f"порядок захвата: съедено {meals2} за 0.5 c — прогресс есть")

## Итоги

1. `Condition.wait()` — **всегда** в цикле `while` по инварианту:
   `notify_all` будит всех, а спящие пробуждения POSIX разрешает.
2. Очереди классифицируются по числу производителей/потребителей
   (SPSC/SPMC/MPSC/MPMC); стандартная `queue.Queue` — **MPMC**
   и решает «не пусто / не переполнено» за вас.
3. GIL: I/O-bound — потоки ускоряют, CPU-bound на Питоне — нет
   (тогда `multiprocessing`).
4. Дедлок — цикл ожиданий; лечится иерархией замков, таймаутами,
   одним замком или очередью-посредником. Ищется через
   `faulthandler.dump_traceback_later`.
5. Лайвлок — потоки активны, но отменяют друг другу шаги; лечится
   теми же приёмами + случайным/экспоненциальным бэкоффом вместо
   синхронных ретраев.
6. Старвация — честные очереди ожидания и таймауты.

Домашка по этой теме — через `queue.Queue` и замки; ручные
`Condition` понадобятся только если понадобится собственный
блокирующий протокол.